# Leave-one-cohort-out (LOCO) splits

Builds the four LOCO training sets used in the paper by removing, in turn,
each of the four REAL-Colon acquisition cohorts from the training
annotations.


Only `instances_train.json` is rewritten. The training dataloader also needs
validation and test annotations, so copy `instances_val.json` and
`instances_test.json` from `data/rc41/annotations` into each
`data/rc41_loco_<cohort>/annotations` folder before training on it.


In [ ]:
import json


In [ ]:
x = json.load(open("data/rc41/annotations/instances_train.json"))


In [ ]:
len(x["images"]), len(x["annotations"])

In [ ]:
allvids = set([x["images"][i]["sequence_id"] for i in range(len(x["images"]))])

partition by cohort to see statistics

In [ ]:
vidbycohort = {
    k: [v for v in allvids if k in v]
    for k in ["001-", "002-", "003-", "004-"]
}
vidbycohortlen = {
    k: len(v) for k, v in vidbycohort.items()
}
print(vidbycohortlen)
allident = set([x["annotations"][i]["identity_id"] for i in range(len(x["annotations"]))])
identbycohort = {
    k: [v for v in allident if k in v]
    for k in ["001-", "002-", "003-", "004-"]
}
print(identbycohort)
identbycohortlen = {
    k: len(v) for k, v in identbycohort.items()
}
print(identbycohortlen)

In [ ]:
def leave_out(x, cohort):
    imgid2imgidx = {x["images"][i]["id"]: i for i in range(len(x["images"]))}
    out = {
        "images": [x["images"][i] for i in range(len(x["images"])) if cohort != x["images"][i]["sequence_id"].split("-")[0]],
        "annotations": [x["annotations"][i] for i in range(len(x["annotations"])) if cohort != x["images"][imgid2imgidx[x["annotations"][i]["image_id"]]]["sequence_id"].split("-")[0]]
    }
    return out

In [ ]:
def stats(ds):
    allvids = set([ds["images"][i]["sequence_id"] for i in range(len(ds["images"]))])
    vidbycohort = {
        k: [v for v in allvids if k in v]
        for k in ["001-", "002-", "003-", "004-"]
    }
    print(vidbycohort)
    vidbycohortlen = {
        k: len(v) for k, v in vidbycohort.items()
    }
    print(vidbycohortlen)
    print(len(ds["images"]), len(ds["annotations"]))
    allident = set([ds["annotations"][i]["identity_id"] for i in range(len(ds["annotations"]))])
    identbycohort = {
        k: [v for v in allident if k in v]
        for k in ["001-", "002-", "003-", "004-"]
    }
    print(identbycohort)
    identbycohortlen = {
        k: len(v) for k, v in identbycohort.items()
    }
    print(identbycohortlen)

In [ ]:
leave_001 = leave_out(x, "001")
leave_002 = leave_out(x, "002")
leave_003 = leave_out(x, "003")
leave_004 = leave_out(x, "004")

In [ ]:
stats(leave_001)

In [ ]:
stats(leave_002)

In [ ]:
stats(leave_003)

In [ ]:
stats(leave_004)

In [ ]:
for cohort, ds in [("001", leave_001), ("002", leave_002), ("003", leave_003), ("004", leave_004)]:
    out = f"data/rc41_loco_{cohort}/annotations/instances_train.json"
    json.dump(ds, open(out, "w"))
